# binary: 가중치 로딩·GPU 직접 추론

전처리가 끝난 그래프 NPZ로 `.pt`를 로딩하는 최소 실행 예제입니다. 이 노트북은 원본 거래 전처리를 실행하지 않습니다. 원본 Parquet부터 실행하려면 `01_binary_v10.ipynb`을 사용하세요.

settings.json의 device를 cuda 또는 cpu로 설정합니다. `AML_DEVICE` 환경변수가 있으면 우선 적용합니다. 기본은 cpu입니다.

In [ ]:
from pathlib import Path
import json

# 노트북 폴더에서 실행하는 것이 기본입니다. 상위 전달 폴더에서도 실행 가능합니다.
ROOT = Path.cwd().resolve()
if not (ROOT / "models" / "binary.pt").is_file():
    ROOT = ROOT / "01_binary_v10"
if not (ROOT / "models" / "binary.pt").is_file():
    raise FileNotFoundError("노트북이 있는 모델 폴더를 작업 디렉터리로 지정하세요.")
FAMILY = "binary"
import os
DEVICE = os.environ.get("AML_DEVICE", json.loads((ROOT / "settings.json").read_text()).get("device", "cpu"))


In [ ]:
import hashlib
import numpy as np
SCHEMA = json.loads((ROOT / "assets/feature_schema.json").read_text())
API_CLASSES = SCHEMA["pattern_classes"]
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from dataclasses import dataclass
import numpy as np
import torch
from torch_geometric.nn import TransformerConv


@dataclass(frozen=True)
class Options:
    neighbors: int = 100
    outgoing: bool = True
    reverse_mp: bool = True
    ego: bool = False
    ports: bool = False
    seed: int = 42


class GraphEncoder(torch.nn.Module):
    """Checkpoint-compatible two-layer encoder with separate message directions."""

    def __init__(self, options):
        super().__init__()
        self.options = options
        node_dim = 19 + int(options.ego)
        edge_dim = 17 + int(options.ports)
        self.conv1 = TransformerConv(
            node_dim, 64, heads=2, edge_dim=edge_dim, dropout=0.0
        )
        self.conv2 = TransformerConv(128, 64, heads=2, edge_dim=edge_dim, dropout=0.0)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(256 + edge_dim, 128), torch.nn.ReLU()
        )
        if options.reverse_mp:
            self.reverse1 = TransformerConv(
                node_dim, 64, heads=2, edge_dim=edge_dim, dropout=0.0, root_weight=False
            )
            self.reverse2 = TransformerConv(
                128, 64, heads=2, edge_dim=edge_dim, dropout=0.0, root_weight=False
            )
            self.merge1 = torch.nn.Linear(256, 128, bias=False)
            self.merge2 = torch.nn.Linear(256, 128, bias=False)

    def encode(
        self, x, edge_index, edge_attr, target_index, target_attr, incoming, outgoing
    ):
        direct, direct_attr = edge_index[:, incoming], edge_attr[incoming]
        reverse, reverse_attr = edge_index[:, outgoing].flip(0), edge_attr[outgoing]
        hidden = self.conv1(x, direct, direct_attr)
        if self.options.reverse_mp:
            hidden = self.merge1(
                torch.cat([hidden, self.reverse1(x, reverse, reverse_attr)], dim=1)
            )
        hidden = hidden.relu()
        encoded = self.conv2(hidden, direct, direct_attr)
        if self.options.reverse_mp:
            encoded = self.merge2(
                torch.cat(
                    [encoded, self.reverse2(hidden, reverse, reverse_attr)], dim=1
                )
            )
        encoded = encoded.relu()
        return self.head(
            torch.cat(
                [encoded[target_index[0]], encoded[target_index[1]], target_attr], dim=1
            )
        )


In [ ]:
class BinaryGNN(GraphEncoder):
    """V10: one independent binary risk head, including unmatched laundering."""

    def __init__(self, masses, options, pos_weight=300.0):
        super().__init__(options)
        self.risk_head = torch.nn.Linear(128, 1)
        self.register_buffer(
            "risk_offset", torch.tensor(-np.log(pos_weight), dtype=torch.float32)
        )

    def forward(self, *batch):
        return self.risk_head(self.encode(*batch)).squeeze(-1)

    def scores(self, *batch):
        return (self(*batch) + self.risk_offset).sigmoid()


In [ ]:
@dataclass
class GraphInput:
    arrays: tuple
    topology: tuple
    edge_id: int
    context_end: str

    def tensors(self, device="cpu"):
        return tuple(
            (
                torch.as_tensor(
                    np.array(a, copy=True),
                    device=device,
                    dtype=(
                        torch.long
                        if i in [1, 3]
                        else torch.bool if i in [5, 6] else torch.float32
                    ),
                )
                for i, a in enumerate(self.arrays)
            )
        )

    def save(self, path):
        np.savez_compressed(
            path,
            **{f"b{i}": v for i, v in enumerate(self.arrays)},
            **{f"t{i}": v for i, v in enumerate(self.topology)},
        )


In [ ]:
def reference_graph(path, metadata):
    with np.load(path, allow_pickle=False) as data:
        return GraphInput(
            tuple((data[f"b{i}"].copy() for i in range(7))),
            tuple((data[f"t{i}"].copy() for i in range(8))),
            int(metadata["edge_id"]),
            metadata["context_end"],
        )


In [ ]:
"""Load trusted packaged state dictionaries and return a backend-friendly prediction."""

import json
import torch


class Predictor:
    """One immutable model instance with validated, explicit decision thresholds."""

    def __init__(self, family, device="cpu"):
        if family != "binary":
            raise ValueError("This notebook only supports binary")
        self.family = family
        self.device = device
        self.path = ROOT / "models" / f"{family}.pt"
        self.meta = json.loads((ROOT / "models" / f"{family}.json").read_text())
        if sha256(self.path) != self.meta["bundle_sha256"]:
            raise ValueError("Model checksum mismatch")
        self.checkpoint = torch.load(self.path, map_location="cpu", weights_only=True)
        self.options = Options(**self.checkpoint["options"])
        if not self.options.ego or self.options.ports:
            raise ValueError("Bundle supports the selected ego/no-ports checkpoints")
        self.model = (
            BinaryGNN(self.checkpoint["train_masses"], self.options).to(device).eval()
        )
        self.model.load_state_dict(self.checkpoint["state_dict"])
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.threshold = self.checkpoint["validation_threshold"]
        torch.set_num_threads(2)

    def predict(self, graph, threshold=None):
        """Return score and decision for one transaction in its declared snapshot."""
        threshold = self.threshold if threshold is None else float(threshold)
        if not 0 < threshold < 1:
            raise ValueError("threshold must be between 0 and 1")
        with torch.no_grad():
            out = self.model.scores(*graph.tensors(self.device))
        risk = float(out[0][0]) if self.family == "nine_class" else float(out[0])
        result = dict(
            model=self.family,
            model_name="이진분류모델" if self.family == "binary" else "9클래스 모델",
            model_version=self.meta["bundle_sha256"][:16],
            edge_id=graph.edge_id,
            context_end=graph.context_end,
            risk_score=risk,
            threshold=threshold,
            alert=risk >= threshold,
        )
        if self.family == "nine_class":
            probabilities = out[1][0].cpu().numpy()
            result.update(
                pattern_class=API_CLASSES[int(probabilities.argmax())],
                pattern_scores={
                    n: float(p) for n, p in zip(API_CLASSES, probabilities)
                },
                pattern_normal_includes_unmatched=True,
            )
        return result


In [ ]:
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available; check the installed PyTorch wheel and driver")
predictor = Predictor(FAMILY, device=DEVICE)
print(
    {
        "model": FAMILY,
        "threshold": predictor.threshold,
        "checkpoint": predictor.meta["bundle_sha256"],
    }
)


In [ ]:
parity_results = []
for name in ["reference_known", "reference_unmatched"]:
    metadata = json.loads((ROOT / "examples" / f"{name}.json").read_text())
    graph = reference_graph(ROOT / "examples" / f"{name}.npz", metadata)
    result = predictor.predict(graph)
    expected = metadata["expected"][FAMILY]
    np.testing.assert_allclose(
        result["risk_score"], expected["risk_score"], rtol=0, atol=2e-5
    )
    if FAMILY == "nine_class":
        np.testing.assert_allclose(
            list(result["pattern_scores"].values()),
            expected["pattern_scores"],
            rtol=0,
            atol=2e-5,
        )
    parity_results.append(
        {
            "fixture": name,
            "passed": True,
            "risk_score": result["risk_score"],
            "expected": expected["risk_score"],
        }
    )
write_json(ROOT / "outputs" / "reference_validation.json", parity_results)
print(parity_results)


In [ ]:
print(json.dumps(result, ensure_ascii=False, indent=2))